In [3]:
import pandas as pd
import numpy as np
from utils.data import load_data, print_random_record
from utils.taxonomy import load_taxonomy
import requests
import json
from openai import OpenAI
import os
from dotenv import load_dotenv

In [ ]:
load_dotenv()
def get_prompt(id, title, abstract):
    wos= id
    prompt= f"""
    Context:
    Following is brief detail building upon GBIF Backbone Taxonomy Dataset. 

    There all biodiversity records are broken into 7 hierarchical levels of taxonomy: [“kingdom”, “phylum”, “class”, “order”, “family”, “genus”, “species”]

    At the very basic level, “kingdom” itself may belong to 9 values: [Plantae, Animalia, Archaea, incertae sedis, Chromista, Bacteria, Protozoa, Viruses, Fungi]

    Task:
    For given “Text” below, you task is to classify the text into different values that can be assigned to 7 levels hierarchy of GBIF Taxonomy. During this task, follow these notes:

    1. Always start with highest taxonomic level of execution. Assign the given “Text” to the kingdom it belongs.
    2. For additional levels after kingdom, infer what it could be. Prioritise internet research, specially from gbif.org as to what it says about it.
    3. If see that some level cannot be inferred because it’s not mentioned, keep value of such level as ‘null’. You may encounter text that may talk broadly about a kingdom and/or phylum but not specific details like genus or species. Or, you may also encounter text where it does not fit into any ‘kingdom’. These are the safe cases where you can use nulls to all applicable 7 keys as null.
    4. If you observe that classification of text may take multiple values of keys in taxonomy, such as a text belonging to multiple phylum, you can attach these values as list to the relevant key.
    5. As you go down the taxonomy, be careful and not attribute value for a level which you don't find supporting evidence from given text. 
    6. You output is a single Json with keys as 7 hierarchical levels of taxonomy and values as list of what you infer in task

    Text:
    {title}:
    {abstract}
    """
    return [wos, prompt]

In [ ]:
# del df, df_sample  # Clean up to save memory
df = pd.read_csv("data/df_sample_6000.csv")
df['prompt'] = df.apply(lambda row: get_prompt(row["UT (Unique WOS ID)"], row['Article Title'], row['Abstract']), axis=1)

# For batch processsing (Select batch manually)
df = df.iloc[1800:2000].copy()

# openai needs batch file in jsonl format
with open("batch_input_1800_2000.jsonl", "w") as f:
    for row in df["prompt"]:
        custom_id, prompt = row
        request_body = {
            "custom_id": custom_id,
            "method": "POST",
            "url": "/v1/chat/completions",
            "body": {
                "model": "o3",
                "messages": [{"role": "user", "content": prompt}],
            }
        }
        f.write(json.dumps(request_body) + "\n")

In [ ]:
api_key = os.getenv("OPENAI_API_KEY")
client = OpenAI(api_key=api_key)

# upload to openai
file = client.files.create(
    file=open("batch_input_1800_2000.jsonl", "rb"),
    purpose="batch"
)
print("Uploaded file ID:", file.id)



Uploaded file ID: file-Cu5s3yX5q5o3Jj8ewj5py4


In [9]:
#submit 
batch = client.batches.create(
    input_file_id=file.id,
    endpoint="/v1/chat/completions",
    completion_window="24h"
)
print("Submitted batch job ID:", batch.id)


Submitted batch job ID: batch_6886646c91788190acd9efb3aef46d23


In [ ]:
import time

while True:
    status = client.batches.retrieve(batch.id)
    print("Status:", status.status)
    if status.status in ["completed", "failed", "cancelled"]:
        break
    time.sleep(60) 


Status: finalizing
Status: finalizing
Status: completed


In [275]:
file_response = client.files.content(status.output_file_id)

In [ ]:
# # Assumes output is downloaded from openai after batch completion
# json_lines = file_response.text.splitlines()
# with open("output_2000_3000.jsonl", "w") as f:
#     for item in json_lines:
#         json_line = json.dumps(item)
#         f.write(json_line + "\n")

# parsed_rows = []
# for line in json_lines:
#     entry = json.loads(line)
#     custom_id = entry.get("custom_id")
#     message = entry.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {})
#     content = message.get("content")
#     tool_calls = message.get("tool_calls", [])
#     parsed_rows.append({
#         "custom_id": custom_id,
#         "content": content,
#         "tool_calls": tool_calls
#     })

# # Convert to DataFrame
# df_results = pd.DataFrame(parsed_rows)